# Correlações
```
Tema: Impacto da IA e das redes sociais na saúde e no desempenho dos estudantes
```
**Pergunta:** o uso de redes sociais e de IA está associado à saúde (sono, bem-estar) e ao desempenho (frequência, atenção, risco acadêmico, nota)?

Usamos **Pearson** (relação linear) e **Spearman** (relação monotônica, robusta a outliers).

---

In [ ]:
import time
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ARQUIVO = "estudantes.parquet"

# >>> Se a detecção automática errar, preencha manualmente (ex.: "horas_uso_ia")
COL_IA = None
COL_NOTA = None

inicial = pl.scan_parquet(ARQUIVO)
esquema = inicial.collect_schema()
colunas = esquema.names()
numericas = [c for c, t in esquema.items() if t.is_numeric()]

def candidatas(tokens, entre):
    return [c for c in entre if set(c.split("_")) & tokens]

cand_ia = candidatas({"ia", "ai", "chatgpt", "llm"}, numericas)
cand_nota = candidatas({"nota", "notas", "gpa", "desempenho", "rendimento"}, numericas)

COL_IA = COL_IA or next((c for c in cand_ia if "horas" in c), cand_ia[0] if cand_ia else None)
COL_NOTA = COL_NOTA or (cand_nota[0] if cand_nota else None)

print("Colunas com 'ia' no nome (qualquer tipo):", candidatas({"ia", "ai", "chatgpt", "llm"}, colunas))
print("Coluna de IA em uso   :", COL_IA)
print("Coluna de nota em uso :", COL_NOTA)
if COL_IA is None:
    print("ATENÇÃO: nenhuma coluna numérica de IA encontrada. Veja a lista 'colunas' e preencha COL_IA.")

# Métricas de saúde e desempenho usadas nos resumos
METRICAS = list(dict.fromkeys(
    c for c in ["horas_redes_sociais", "horas_sono", "pontuacao_vicio_digital", "indice_bem_estar",
                "taxa_frequencia_aulas", "capacidade_atencao_minutos", "pontuacao_risco_academico",
                COL_IA, COL_NOTA]
    if c and c in colunas
))

def perfil(lf, grupo):
    """Resumo por grupo: nº de estudantes + média das métricas do tema."""
    return (
        lf.group_by(grupo)
        .agg([pl.len().alias("estudantes")] +
             [pl.col(c).mean().round(2).alias(f"media_{c}") for c in METRICAS])
        .sort(grupo)
    )


def categorizar(lf):
    """Cria categorias de sono, redes sociais e IA (IA em tercis)."""
    exprs = [
        pl.when(pl.col("horas_sono") <= 6).then(pl.lit("1-Baixo (≤6h)"))
          .when(pl.col("horas_sono") < 8).then(pl.lit("2-Médio (6-8h)"))
          .otherwise(pl.lit("3-Bom (≥8h)")).alias("categoria_sono"),
        pl.when(pl.col("horas_redes_sociais") < 2).then(pl.lit("1-Baixo (<2h)"))
          .when(pl.col("horas_redes_sociais") < 5).then(pl.lit("2-Moderado (2-5h)"))
          .otherwise(pl.lit("3-Alto (≥5h)")).alias("categoria_redes"),
    ]
    if COL_IA:
        exprs.append(
            pl.when(pl.col(COL_IA) <= pl.col(COL_IA).quantile(0.33)).then(pl.lit("1-Baixo uso de IA"))
              .when(pl.col(COL_IA) <= pl.col(COL_IA).quantile(0.66)).then(pl.lit("2-Médio uso de IA"))
              .otherwise(pl.lit("3-Alto uso de IA")).alias("categoria_ia")
        )
    return lf.with_columns(exprs)

In [ ]:
preditores = [c for c in ["horas_redes_sociais", COL_IA, "pontuacao_vicio_digital"] if c and c in colunas]
saude = [c for c in ["horas_sono", "indice_bem_estar"] if c in colunas]
desempenho = [c for c in ["taxa_frequencia_aulas", "capacidade_atencao_minutos", "pontuacao_risco_academico", COL_NOTA] if c and c in colunas]
alvos = saude + desempenho

todas = list(dict.fromkeys(preditores + alvos))
df = inicial.select(todas).drop_nulls().collect()
print(f"{df.height:,} estudantes | preditores: {preditores}")
print(f"saúde: {saude} | desempenho: {desempenho}")

### 1. Matriz de correlação (Pearson)
---

In [ ]:
matriz = pd.DataFrame(df.corr().to_numpy(), index=todas, columns=todas)

plt.figure(figsize=(10, 8))
sns.heatmap(matriz, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlação de Pearson: redes sociais, IA, saúde e desempenho")
plt.tight_layout()
plt.show()

### 2. Preditores x resultados (Pearson e Spearman)
```
|r| < 0,10 desprezível | < 0,30 fraca | < 0,50 moderada | >= 0,50 forte
```
---

In [ ]:
def forca(r):
    a = abs(r)
    return "desprezível" if a < 0.1 else "fraca" if a < 0.3 else "moderada" if a < 0.5 else "forte"

linhas = []
for p in preditores:
    for a in alvos:
        if p == a:
            continue
        r = df.select(
            pl.corr(p, a, method="pearson").alias("pearson"),
            pl.corr(p, a, method="spearman").alias("spearman"),
        ).row(0)
        linhas.append({"preditor": p, "alvo": a, "pearson": r[0], "spearman": r[1],
                       "r2": r[0] ** 2, "forca": forca(r[0]),
                       "direcao": "positiva" if r[0] > 0 else "negativa"})

tabela = pl.DataFrame(linhas).with_columns(pl.col("pearson").abs().alias("abs_r")).sort("abs_r", descending=True)
pl.Config.set_tbl_rows(50)
display(tabela.drop("abs_r"))

In [ ]:
pivot = tabela.to_pandas().pivot(index="preditor", columns="alvo", values="pearson")

plt.figure(figsize=(10, 4 + len(preditores)))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Preditores (linhas) x Saúde e Desempenho (colunas)")
plt.tight_layout()
plt.show()

### 3. Dispersão dos 3 pares mais fortes
```
Amostra de 5.000 estudantes (500 mil pontos ilegíveis) com reta de tendência.
```
---

In [ ]:
amostra = df.sample(n=min(5000, df.height), seed=42).to_pandas()
top = tabela.head(3).select(["preditor", "alvo", "pearson"]).rows()

fig, axes = plt.subplots(1, len(top), figsize=(6 * len(top), 5), squeeze=False)
for ax, (p, a, r) in zip(axes[0], top):
    sns.regplot(data=amostra, x=p, y=a, ax=ax,
                scatter_kws={"alpha": 0.15, "s": 8}, line_kws={"color": "red"})
    ax.set_title(f"{p} x {a}\nr = {r:.2f}")
plt.tight_layout()
plt.show()

### 4. Médias por nível de uso
```
Visão mais intuitiva: o que acontece com saúde e desempenho conforme o uso aumenta?
```
---

In [ ]:
lf_cat = categorizar(inicial)
grupos = ["categoria_redes"] + (["categoria_ia"] if COL_IA else [])
resumo_alvos = [c for c in ["indice_bem_estar", "horas_sono", "taxa_frequencia_aulas", "capacidade_atencao_minutos"] if c in colunas]

fig, axes = plt.subplots(len(grupos), len(resumo_alvos), figsize=(5 * len(resumo_alvos), 4 * len(grupos)), squeeze=False)
for i, g in enumerate(grupos):
    medias = (
        lf_cat.group_by(g)
        .agg([pl.col(c).mean().alias(c) for c in resumo_alvos])
        .sort(g).collect().to_pandas()
    )
    for j, a in enumerate(resumo_alvos):
        sns.barplot(data=medias, x=g, y=a, ax=axes[i][j], color="steelblue")
        axes[i][j].set_title(f"Média de {a}")
        axes[i][j].set_xlabel("")
        axes[i][j].tick_params(axis="x", rotation=15)
        lo, hi = medias[a].min(), medias[a].max()
        margem = (hi - lo) * 0.5 or 1
        axes[i][j].set_ylim(lo - margem, hi + margem)   # eixo ampliado para mostrar a diferença
plt.tight_layout()
plt.show()

### 5. Conclusões automáticas
---

In [ ]:
print("Principais associações (ordenadas por |r|):\n")
for p, a, r, s, f, d in tabela.select(["preditor", "alvo", "pearson", "spearman", "forca", "direcao"]).head(8).rows():
    print(f"- {p} x {a}: Pearson {r:+.3f} | Spearman {s:+.3f} -> correlação {f} {d}")

print("\nSem associação relevante (|r| < 0,10):")
fracas = tabela.filter(pl.col("abs_r") < 0.1).select(["preditor", "alvo"]).rows()
print(", ".join(f"{p} x {a}" for p, a in fracas) or "nenhuma")

### Como interpretar (escreva sua conclusão com os números acima)
- **Força:** com 500 mil linhas qualquer correlação é "estatisticamente significativa"; o que importa é o **tamanho** de r.
- **r² :** percentual da variação de um indicador que a outra variável explica linearmente.
- **Vício digital:** a pontuação provavelmente é construída a partir do uso de redes/IA, então a correlação alta com elas é esperada e não é um achado independente.
- **Correlação não é causa:** sono, escolaridade e país podem influenciar tudo ao mesmo tempo. A regressão múltipla (próximo notebook) ajuda a separar os efeitos.